# Pressures → sacks linear regression

Manual notebook for Kevin Zhang (BDB 2027). Fits a simple OLS of **n_sacks ~ n_pressures** at season × player grain.

## Prerequisites (Kevin runs this locally)

1. Generate DerivedData first via `notebooks/cursor_dev/player_play_passrush_aggs.ipynb`.
2. Confirm this file exists relative to repo root:
   `DerivedData/player_play_passrush_by_season_player.csv`
3. Then run **this** notebook.

**Do not** paste DerivedData CSV contents into agent chat or Cursor context. `DerivedData/` is cursorignored — agents should not load or summarize the raw file.

This notebook is intentionally left unexecuted in git; Kevin runs it after DerivedData is present.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Paths relative to repo root (notebook lives in notebooks/cursor_dev/)
REPO_ROOT = Path("../..").resolve()
IN_CSV = REPO_ROOT / "DerivedData" / "player_play_passrush_by_season_player.csv"

print("REPO_ROOT:", REPO_ROOT)
print("Input:", IN_CSV)
print("Exists:", IN_CSV.is_file())

## Load + NA handling

Drop rows missing `n_pressures` or `n_sacks` before fitting.

In [ ]:
assert IN_CSV.is_file(), (
    f"Missing {IN_CSV}. Run player_play_passrush_aggs.ipynb first to build DerivedData."
)

df = pd.read_csv(IN_CSV)
print("Loaded shape:", df.shape)
print("Columns:", list(df.columns))

needed = ["n_pressures", "n_sacks"]
missing_cols = [c for c in needed if c not in df.columns]
assert not missing_cols, f"CSV missing columns: {missing_cols}"

before = len(df)
fit_df = df.dropna(subset=needed).copy()
fit_df["n_pressures"] = pd.to_numeric(fit_df["n_pressures"], errors="coerce")
fit_df["n_sacks"] = pd.to_numeric(fit_df["n_sacks"], errors="coerce")
fit_df = fit_df.dropna(subset=needed)
print(f"Rows: {before} → {len(fit_df)} after dropping NA on {needed}")

## Fit: `n_sacks ~ n_pressures`

Simple sklearn `LinearRegression` (OLS with intercept).

In [ ]:
X = fit_df[["n_pressures"]].to_numpy(dtype=float)
y = fit_df["n_sacks"].to_numpy(dtype=float)

model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

coef = float(model.coef_[0])
intercept = float(model.intercept_)
r2 = float(r2_score(y, y_hat))

print(f"n_sacks = {intercept:.6f} + {coef:.6f} * n_pressures")
print(f"coef (n_pressures): {coef:.6f}")
print(f"intercept:          {intercept:.6f}")
print(f"R²:                 {r2:.6f}")
print(f"n:                  {len(fit_df)}")

## Scatter + fitted line

In [ ]:
x = fit_df["n_pressures"].to_numpy(dtype=float)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = intercept + coef * x_line

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, alpha=0.35, s=18, edgecolors="none", label="season × player")
ax.plot(x_line, y_line, color="C1", linewidth=2, label=f"fit (R²={r2:.3f})")
ax.set_xlabel("n_pressures")
ax.set_ylabel("n_sacks")
ax.set_title("n_sacks ~ n_pressures")
ax.legend()
fig.tight_layout()
plt.show()